# 1 · Track the body in a video

**What this notebook does:** It goes through your video one frame (one still picture) at a time and uses
**MediaPipe**, a free body-tracking tool from Google, to find 33 points on the speaker's body: shoulders,
elbows, wrists, fingers, nose, eyes and so on. For every frame it writes down where each point is.

| You need | You get |
|---|---|
| A video in the `Videos` folder. The workshop video `Test_Video.MOV` is already there. | `Keypoints/Test_Video_keypoints.csv`: a table with one row per frame and the position of every body point. |
| | `Keypoints/Test_Video_overlay.mp4`: your video with the tracked skeleton drawn on top, so you can check the tracking by eye. |

<img src="../docs/keypoints.png" alt="Body points found by MediaPipe" width="400">

### New to notebooks? Read this first

A notebook mixes text (like this) with grey **code cells**.

* **Run a cell:** click on it and press **Shift + Enter**. The next cell gets selected, so pressing
  Shift + Enter again and again walks you through the notebook step by step.
* While a cell is running you see `[*]` on its left. When it is done, a number appears instead, such as `[4]`.
* **Run everything at once:** menu *Run → Run All Cells*.
* **Start over:** menu *Kernel → Restart Kernel and Clear Outputs of All Cells…*, then run from the top again.
* Inside code cells, everything after a `#` is a comment: a note for you, not an instruction for the computer.
* You only ever need to change the **Settings** cell. Everything else works as it is.

## Step 1 · Load the tools

Python on its own can't read videos or track bodies. These lines load the add-on packages (called *libraries*) that can.

In [ ]:
from pathlib import Path        # works with files and folders
import urllib.request           # downloads files from the internet

import cv2                      # OpenCV: reads and writes videos
import mediapipe as mp          # Google's body tracking
from mediapipe.tasks.python import BaseOptions, vision
import pandas as pd             # tables, like a spreadsheet
import plotly.express as px     # interactive plots
from tqdm.auto import tqdm      # progress bars

## Step 2 · Settings

This is the only cell you need to change. Text in quotes (`"..."`) must stay in quotes.

In [ ]:
# The name of your video in the Videos folder, without the ending (.MOV, .mp4, ...)
VIDEO_NAME = "Test_Video"

# Which tracking model to use:
#   "lite"  = fastest, a little less accurate (good for older laptops)
#   "full"  = a good balance (recommended)
#   "heavy" = most accurate, but about 3 times slower
MODEL = "full"

# Also make a copy of the video with the tracked skeleton drawn on it? (True or False)
SAVE_OVERLAY_VIDEO = True

# How sure MediaPipe must be (from 0 to 1) before it reports a person.
# Lower it if the person is often "not found"; raise it if it finds people who aren't there.
MIN_CONFIDENCE = 0.4

## Step 3 · Find the video and the tracking model

The tracking *model* is the trained "brain" that MediaPipe uses to recognise bodies. The `full` model comes
with the workshop files. `lite` and `heavy` are downloaded automatically the first time you choose them
(this needs an internet connection).

In [ ]:
VIDEOS_FOLDER = Path("../Videos")
KEYPOINTS_FOLDER = Path("../Keypoints")
KEYPOINTS_FOLDER.mkdir(exist_ok=True)

# Find the video, whatever its ending is (.MOV, .mp4, .avi, ...)
matches = sorted(VIDEOS_FOLDER.glob(VIDEO_NAME + ".*"))
if not matches:
    videos_there = sorted(p.stem for p in VIDEOS_FOLDER.iterdir() if p.is_file())
    raise FileNotFoundError(f"There is no video called '{VIDEO_NAME}' in the Videos folder. "
                            f"These videos are there: {videos_there}")
video_path = matches[0]

# Get the tracking model (downloaded only the first time)
model_path = Path(f"pose_landmarker_{MODEL}.task")
if not model_path.exists():
    print(f"Downloading the '{MODEL}' model. This only happens once...")
    url = ("https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
           f"pose_landmarker_{MODEL}/float16/latest/pose_landmarker_{MODEL}.task")
    urllib.request.urlretrieve(url, model_path)

# Read some basic facts about the video
video = cv2.VideoCapture(str(video_path))
fps = video.get(cv2.CAP_PROP_FPS)                        # frames per second
frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
video.release()
if fps == 0:
    raise ValueError(f"Could not open {video_path.name}. Try converting it to .mp4 first.")

print(f"Video: {video_path.name}")
print(f"{frame_count} frames, {fps:.2f} frames per second, {width} x {height} pixels, "
      f"{frame_count / fps:.1f} seconds long")
print(f"Model: {model_path.name}")

## Step 4 · Try the tracking on one frame

Before tracking the whole video, let's check that it works on one frame from the middle of the video.

First we set up a few things we'll use again in Step 5:
* the names of MediaPipe's 33 body points, for example `nose`, `right_wrist` or `left_index` (= left index finger),
* a small *function* (a reusable recipe) that hands a video frame to MediaPipe, and one that draws the skeleton
  (OpenCV writes colours as Blue-Green-Red, so `(0, 0, 255)` is red),
* the tracker's options.

In [ ]:
POINT_NAMES = [point.name.lower() for point in vision.PoseLandmark]
print(len(POINT_NAMES), "body points:", ", ".join(POINT_NAMES))


def to_mediapipe_image(frame):
    """OpenCV stores colours as Blue-Green-Red, MediaPipe wants Red-Green-Blue."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)


# How the skeleton looks: red dots joined by white lines
POINT_STYLE = vision.drawing_utils.DrawingSpec(color=(0, 0, 255), thickness=-1, circle_radius=5)
LINE_STYLE = vision.drawing_utils.DrawingSpec(color=(255, 255, 255), thickness=3)


def draw_skeleton(frame, body):
    """Draw the tracked points, and the lines between them, onto a video frame."""
    vision.drawing_utils.draw_landmarks(frame, body, vision.PoseLandmarksConnections.POSE_LANDMARKS,
                                        POINT_STYLE, LINE_STYLE)


options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(model_path)),
    running_mode=vision.RunningMode.VIDEO,  # each frame helps to find the body in the next one
    min_pose_detection_confidence=MIN_CONFIDENCE,
    min_pose_presence_confidence=MIN_CONFIDENCE,
    min_tracking_confidence=MIN_CONFIDENCE,
)

In [ ]:
# Jump to the middle of the video and take one frame
video = cv2.VideoCapture(str(video_path))
video.set(cv2.CAP_PROP_POS_FRAMES, frame_count // 2)
ok, frame = video.read()
video.release()

# Track the body in that frame and draw the skeleton on it
with vision.PoseLandmarker.create_from_options(options) as tracker:
    result = tracker.detect_for_video(to_mediapipe_image(frame), 0)

if result.pose_landmarks:
    draw_skeleton(frame, result.pose_landmarks[0])
else:
    print("No person was found in this frame.")

fig = px.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), binary_format="jpg",
                title="One frame with the tracked skeleton")
fig.show()

**What to look for:** red dots on the joints, connected by white lines. The picture is interactive: drag across
it to zoom in, double-click to zoom back out.

If the skeleton sits on the person, you're ready for the next step. If it's missing or in the wrong place, try
`MODEL = "heavy"` in the Settings, or a video where the whole upper body is visible and well lit.

## Step 5 · Track every frame

This is the slow step: MediaPipe looks at every frame of the video, one after the other. The progress bar shows
how far along it is and about how long is left. The 85-second workshop video takes 1 to 3 minutes on a typical laptop.

Good to know:
* Positions are saved as numbers between 0 and 1: **x** goes from 0 (left edge of the video) to 1 (right edge),
  **y** from 0 (top edge) to 1 (bottom edge).
* "Right" and "left" mean the **speaker's** right and left. If they face the camera, their right hand is on the
  left side of the picture.
* If no person is found in a frame, that row is left empty. Notebook 2 fills in these gaps.

In [ ]:
rows = []  # we collect one row per video frame here

video = cv2.VideoCapture(str(video_path))
if SAVE_OVERLAY_VIDEO:
    overlay_path = KEYPOINTS_FOLDER / f"{VIDEO_NAME}_overlay.mp4"
    mp4_format = cv2.VideoWriter_fourcc(*"mp4v")
    overlay = cv2.VideoWriter(str(overlay_path), mp4_format, fps, (width, height))

with vision.PoseLandmarker.create_from_options(options) as tracker:
    for frame_number in tqdm(range(frame_count), unit="frame"):
        ok, frame = video.read()
        if not ok:  # no more frames
            break
        time_ms = frame_number * 1000 / fps  # when this frame is shown, in milliseconds
        result = tracker.detect_for_video(to_mediapipe_image(frame), round(time_ms))

        row = {"time_ms": time_ms}
        if result.pose_landmarks:                # a person was found in this frame
            body = result.pose_landmarks[0]      # the 33 points of the (first) person
            for name, point in zip(POINT_NAMES, body):
                row[name + "_x"] = point.x
                row[name + "_y"] = point.y
            if SAVE_OVERLAY_VIDEO:
                draw_skeleton(frame, body)
        rows.append(row)

        if SAVE_OVERLAY_VIDEO:
            overlay.write(frame)

video.release()
if SAVE_OVERLAY_VIDEO:
    overlay.release()
    print("Saved the overlay video:", overlay_path)

## Step 6 · Save the results

We turn the collected rows into a table and save it as a `.csv` file, which you can also open in Excel, R or SPSS.

In [ ]:
columns = ["time_ms"] + [name + axis for name in POINT_NAMES for axis in ["_x", "_y"]]
keypoints = pd.DataFrame(rows, columns=columns)

keypoints_path = KEYPOINTS_FOLDER / f"{VIDEO_NAME}_keypoints.csv"
keypoints.to_csv(keypoints_path, index=False)

found = keypoints["nose_x"].notna().mean() * 100
print(f"A person was found in {found:.1f}% of the frames.")
print("Saved:", keypoints_path)
keypoints.head()

Above are the first rows of the saved table. Each row is one video frame. `time_ms` is when the frame appears
(in milliseconds; 1000 ms = 1 second), followed by an `_x` and a `_y` column for every body point.

## Step 7 · Look at the movement

This plot shows the height of both wrists over the whole video. When a hand goes up for a gesture, its line goes up.

* Drag across the plot to zoom in on a stretch of time; double-click to zoom back out.
* Click a name in the legend to hide or show that line.

In [ ]:
fig = px.line(keypoints, x="time_ms", y=["right_wrist_y", "left_wrist_y"],
              title="Height of the wrists (higher in the plot = hand higher in the video)",
              labels={"time_ms": "Time (ms)", "value": "Position (0 = top of the video)",
                      "variable": ""})
fig.update_yaxes(autorange="reversed")  # in videos y counts from the top down, so we flip the axis
fig.show()

## Done!

* Open `Keypoints/Test_Video_overlay.mp4` in a video player to watch the tracking.
* Continue with **Notebook 2**, which turns these positions into speeds.